# 00 - Environment Setup and Data Loading

This notebook sets up the Google Colab environment, installs dependencies, clones the project repository, and verifies all data files are accessible.

**Runtime:** ~2 minutes

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm geopandas folium
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally - detecting project root...")
    # Robust local path detection: walk up from notebook location to find project root
    # Works whether launched from notebook dir, project root, or anywhere else
    _nb_dir = os.getcwd()
    _candidate = _nb_dir
    PROJECT_ROOT = None
    for _ in range(10):  # Walk up at most 10 levels
        if os.path.isfile(os.path.join(_candidate, 'requirements.txt')) and \
           os.path.isdir(os.path.join(_candidate, 'src', 'ems_readiness')):
            PROJECT_ROOT = _candidate
            break
        _parent = os.path.dirname(_candidate)
        if _parent == _candidate:
            break
        _candidate = _parent
    if PROJECT_ROOT is None:
        # Fallback: assume notebook is at notebooks/colab_standalone/individual/
        PROJECT_ROOT = os.path.abspath(os.path.join(_nb_dir, '..', '..', '..'))
    print(f"  Project root: {PROJECT_ROOT}")

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

## Data Acquisition

The raw crash data (~536 MB) is too large for Git. The cell below downloads it from NYC Open Data
and runs the processing pipeline to generate `crashes_manhattan.csv` and other processed files.

**First run may take 10-20 minutes.** Subsequent runs skip if files exist.

In [ ]:
import glob

sys.path.insert(0, PROJECT_ROOT)

crashes_csv = os.path.join(PROCESSED_DIR, 'crashes_manhattan.csv')
raw_crash_pattern = os.path.join(RAW_DIR, 'Motor_Vehicle_Collisions_-_Crashes_*.csv')

if not os.path.exists(crashes_csv):
    print('crashes_manhattan.csv not found -- running data pipeline...\n')
    if not glob.glob(raw_crash_pattern):
        print('Downloading raw crash data from NYC Open Data...')
        from scripts.download_crash_data import download_crash_data
        download_crash_data()
        print()
    print('Running data processing pipeline...')
    from scripts.generate_all_data import ensure_data
    ensure_data(project_root=PROJECT_ROOT, force=True)
    print('Data pipeline complete.')
else:
    print(f'crashes_manhattan.csv found ({os.path.getsize(crashes_csv)/1e6:.1f} MB) -- skipping.')

## Verify Data Files

Check that all required raw and processed data files exist.

In [ ]:
import glob

# Check raw data files
raw_files = {
    'FDNY Firehouses': 'FDNY_Firehouse_Listing_*.csv',
    'Manhattan Boundary': 'manhattan_boundary.geojson',
    'CBD Boundary': 'cbd_boundary.geojson',
    'NYC Boundaries': 'nyc_borough_boundaries.geojson',
    'Motor Vehicle Collisions': 'Motor_Vehicle_Collisions_-_Crashes_*.csv',
    'Police Precincts': 'Police_Precincts_*.csv',
}

print("=== RAW DATA FILES ===")
for name, pattern in raw_files.items():
    matches = glob.glob(os.path.join(RAW_DIR, pattern))
    status = "FOUND" if matches else "MISSING"
    print(f"  {status}: {name}")
    if matches:
        for m in matches:
            size_mb = os.path.getsize(m) / 1e6
            print(f"         {os.path.basename(m)} ({size_mb:.1f} MB)")

# Check processed data files
processed_files = [
    'crashes_manhattan.csv',
    'firehouses_manhattan.csv',
    'firehouses_clean.csv',
    'distance_matrix_firehouse_precinct.csv',
    'distance_matrix_firehouse_precinct_manhattan.csv',
    'demand_lambda_hourly.csv',
    'demand_lambda_dow.csv',
    'demand_lambda_precinct.csv',
]

print("\n=== PROCESSED DATA FILES ===")
for f in processed_files:
    path = os.path.join(PROCESSED_DIR, f)
    status = "FOUND" if os.path.exists(path) else "MISSING"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {status}: {f} ({size_mb:.1f} MB)")
    else:
        print(f"  {status}: {f}")

## Verify Configuration Files

In [ ]:
import yaml

config_files = ['demand.yaml', 'service.yaml', 'optimization.yaml', 'simulation.yaml']
print("=== CONFIGURATION FILES ===")
for cf in config_files:
    path = os.path.join(CONFIGS_DIR, cf)
    if os.path.exists(path):
        with open(path) as f:
            config = yaml.safe_load(f)
        print(f"  FOUND: {cf}")
        for key, val in config.items():
            if not isinstance(val, dict):
                print(f"         {key}: {val}")
    else:
        print(f"  MISSING: {cf}")

## Test Package Imports

In [ ]:
try:
    from ems_readiness.utils.distance import haversine, manhattan_distance, build_distance_matrix
    print("utils.distance: OK")
except ImportError as e:
    print(f"utils.distance: FAILED - {e}")

try:
    from ems_readiness.service.travel_time import travel_time_minutes, build_travel_time_matrix
    print("service.travel_time: OK")
except ImportError as e:
    print(f"service.travel_time: FAILED - {e}")

try:
    from ems_readiness.service.service_time import ServiceTimeModel
    print("service.service_time: OK")
except ImportError as e:
    print(f"service.service_time: FAILED - {e}")

try:
    from ems_readiness.demand.arrival_generator import NHPPArrivalGenerator
    print("demand.arrival_generator: OK")
except ImportError as e:
    print(f"demand.arrival_generator: FAILED - {e}")

try:
    from ems_readiness.optimization.models import build_demand_weighted, solve_model
    from ems_readiness.optimization.policies import uniform_allocation, demand_proportional_allocation
    print("optimization: OK")
except ImportError as e:
    print(f"optimization: FAILED - {e}")

try:
    from ems_readiness.simulation.engine import EMSSimulation
    from ems_readiness.simulation.runner import BatchRunner
    print("simulation: OK")
except ImportError as e:
    print(f"simulation: FAILED - {e}")

print("\nAll imports verified successfully!")

## Quick Data Preview

In [ ]:
# Load and preview key datasets
if os.path.exists(os.path.join(PROCESSED_DIR, 'crashes_manhattan.csv')):
    crashes = pd.read_csv(os.path.join(PROCESSED_DIR, 'crashes_manhattan.csv'), nrows=5)
    print("Crashes (first 5 rows):")
    display(crashes.head())

if os.path.exists(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv')):
    fh = pd.read_csv(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv'))
    print(f"\nManhattan Firehouses: {len(fh)} locations")
    display(fh[['FacilityName', 'Latitude', 'Longitude', 'in_cbd']].head(10))

if os.path.exists(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv')):
    prec = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv'))
    print(f"\nPrecinct demand rates: {len(prec)} precincts")
    display(prec.head(10))

print("\nSetup complete! Proceed to subsequent notebooks.")